# Lab 9: Named Entity Recognition (NER)

In [1]:
import sys
import subprocess
from collections import Counter

import spacy
from spacy import displacy
from spacy.training import Example

# download the small English model only if it is missing
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

nlp = spacy.load("en_core_web_sm")

## Task 1

In [2]:
# 1. ask the user to enter a sentence
try:
    sentence = input("Enter a sentence: ")
except Exception:          # no keyboard input available (notebook run non-interactively)
    sentence = ""

if not sentence.strip():   # empty input -> sample sentence
    sentence = "Sundar Pichai, the CEO of Google, announced the new phone in New York on Tuesday for $799."
    print("(no input given, a sample sentence is used)")

doc = nlp(sentence)
print("Sentence:", sentence)
print("Entities recognized in text:", doc.ents)

(no input given, a sample sentence is used)
Sentence: Sundar Pichai, the CEO of Google, announced the new phone in New York on Tuesday for $799.
Entities recognized in text: (Sundar Pichai, Google, New York, Tuesday, 799)


In [3]:
# 2. details of each entity
for ent in doc.ents:
    print(f"Entity: {ent.text}")
    print(f"Label: {ent.label_}")
    print(f"Start Position: {ent.start}")
    print(f"End Position: {ent.end}")
    print(f"Entity Type: {spacy.explain(ent.label_)}")
    print("")

Entity: Sundar Pichai
Label: PERSON
Start Position: 0
End Position: 2
Entity Type: People, including fictional

Entity: Google
Label: ORG
Start Position: 6
End Position: 7
Entity Type: Companies, agencies, institutions, etc.

Entity: New York
Label: GPE
Start Position: 13
End Position: 15
Entity Type: Countries, cities, states

Entity: Tuesday
Label: DATE
Start Position: 16
End Position: 17
Entity Type: Absolute or relative dates or periods

Entity: 799
Label: MONEY
Start Position: 19
End Position: 20
Entity Type: Monetary values, including unit



## Task 2

In [4]:
text = "This agreement is made and entered into by and between ‘Abc & Co.’  and ‘Bcd LLC’ for term 1 year starting from April 1, 2020,  hereinafter collectively referred to as the Parties. Samuel P. Jackson in the place (New York) and on the date written below, with the following terms and conditions."

doc = nlp(text)

print("Entities found by the model:")
for ent in doc.ents:
    print(f"  {ent.text!r:<22} {ent.label_}")

Entities found by the model:
  'between ‘Abc & Co.'   ORG
  '1 year'               DATE
  'April 1, 2020'        DATE
  'Samuel P. Jackson'    PERSON
  'New York'             GPE


### 1. Entities that were not identified

In this run the model **missed `Bcd LLC`** (it should be an `ORG`), and its span for `Abc & Co.` is wrong (it wrongly starts with `between ‘`). Both are added / corrected below.

In [5]:
from spacy.tokens import Span

missing = [("Abc & Co.", "ORG"), ("Bcd LLC", "ORG")]

new_ents = []
for entity_text, label in missing:
    start = text.find(entity_text)
    new_ents.append(doc.char_span(start, start + len(entity_text), label=label))

# keep the model's entities, except the ones that overlap with the new (corrected) ones
kept = [e for e in doc.ents if not any(e.start < n.end and n.start < e.end for n in new_ents)]
doc.ents = kept + new_ents

for ent in doc.ents:
    print(f"  {ent.text!r:<22} {ent.label_}")

  'Abc & Co.'            ORG
  'Bcd LLC'              ORG
  '1 year'               DATE
  'April 1, 2020'        DATE
  'Samuel P. Jackson'    PERSON
  'New York'             GPE


### 2. Count of `PERSON` and `ORG`

In [6]:
counts = Counter(ent.label_ for ent in doc.ents)

print("PERSON:", counts["PERSON"], [e.text for e in doc.ents if e.label_ == "PERSON"])
print("ORG   :", counts["ORG"], [e.text for e in doc.ents if e.label_ == "ORG"])

PERSON: 1 ['Samuel P. Jackson']
ORG   : 2 ['Abc & Co.', 'Bcd LLC']


### 3. Visualize the entities

In [7]:
displacy.render(doc, style="ent", jupyter=True)

## Task 3: Training a Custom NER Model

### 1. Create a training example with `Example.from_dict()`

In [8]:
text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products."

annotations = [("Microsoft", "ORG"),
               ("Bill Gates", "PERSON"),
               ("Paul Allen", "PERSON"),
               ("Albuquerque", "GPE"),
               ("New Mexico", "GPE"),
               ("Windows", "PRODUCT"),
               ("Office", "PRODUCT")]

entities = [(text.find(t), text.find(t) + len(t), label) for t, label in annotations]

# fine-tune the pre-trained model
spacy.util.fix_random_seed(42)
nlp = spacy.load("en_core_web_sm")
ner = nlp.get_pipe("ner")
for label in {label for _, label in annotations}:
    ner.add_label(label)

example = Example.from_dict(nlp.make_doc(text), {"entities": entities})

print("Training example entities:")
for start, end, label in entities:
    print(f"  {text[start:end]!r:<26} {label}")

Training example entities:
  'Microsoft'                ORG
  'Bill Gates'               PERSON
  'Paul Allen'               PERSON
  'Albuquerque'              GPE
  'New Mexico'               GPE
  'Windows'                  PRODUCT
  'Office'                   PRODUCT


### 2. Train the NER model for 20 iterations

In [9]:
with nlp.select_pipes(enable="ner"):          # only the NER component is updated
    optimizer = nlp.resume_training()
    for iteration in range(20):
        losses = {}
        nlp.update([example], drop=0.5, sgd=optimizer, losses=losses)
        print(f"Iteration {iteration + 1:2d} - NER loss: {losses['ner']:.4f}")

Iteration  1 - NER loss: 5.3574
Iteration  2 - NER loss: 3.4425
Iteration  3 - NER loss: 5.9476
Iteration  4 - NER loss: 3.2706
Iteration  5 - NER loss: 3.8379
Iteration  6 - NER loss: 5.0203
Iteration  7 - NER loss: 1.7127


Iteration  8 - NER loss: 3.3791
Iteration  9 - NER loss: 2.1028


Iteration 10 - NER loss: 1.6577
Iteration 11 - NER loss: 2.1830
Iteration 12 - NER loss: 2.7752
Iteration 13 - NER loss: 5.6960
Iteration 14 - NER loss: 0.1799
Iteration 15 - NER loss: 1.3524
Iteration 16 - NER loss: 0.6301


Iteration 17 - NER loss: 0.0933
Iteration 18 - NER loss: 0.0983


Iteration 19 - NER loss: 0.7380
Iteration 20 - NER loss: 0.3214


### 3. Save the trained model in a folder named `custom_ner_model`

In [10]:
nlp.to_disk("custom_ner_model")
print("Model saved in the folder: custom_ner_model")

Model saved in the folder: custom_ner_model


### 5. Load the saved model and identify named entities

In [11]:
custom_nlp = spacy.load("custom_ner_model")

sentences = [text,
             "Bill Gates and Paul Allen started Microsoft in New Mexico.",
             "Steve Jobs and Steve Wozniak founded Apple in Cupertino, California."]

for sentence in sentences:
    doc = custom_nlp(sentence)
    print(sentence)
    for ent in doc.ents:
        print(f"   {ent.text:<15} {ent.label_}")
    print()

Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products.
   Microsoft       ORG
   Bill Gates      PERSON
   Paul Allen      PERSON
   Albuquerque     GPE
   New Mexico      GPE
   Windows         PRODUCT
   Office          PRODUCT

Bill Gates and Paul Allen started Microsoft in New Mexico.
   Bill Gates      PERSON
   Paul Allen      PERSON
   Microsoft       ORG
   New Mexico      GPE

Steve Jobs and Steve Wozniak founded Apple in Cupertino, California.
   Steve Jobs      PERSON
   Steve Wozniak   PERSON
   Apple           ORG
   Cupertino       GPE
   California      GPE

